# F2-M03: Análisis de Valores Nulos

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Análisis exhaustivo de valores faltantes: porcentaje global, clasificación
por severidad, patrón de nulidad y correlación entre columnas nulas.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m03_nulos.html`
- `docs/html/fase2/graficos/m03_donut_clasificacion.html`
- `docs/html/fase2/graficos/m03_barras_nulos.html`
- `docs/html/fase2/graficos/m03_patron_nulos.html`
- `docs/html/fase2/graficos/m03_correlacion_nulos.html`

## Flujo
```
... → M02 Calidad → **M03 Nulos** → M04 Univariante Num → ...
```

## Siguiente
`f2_m04_univariante_num.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    _cwd = Path.cwd()
    ROOT = _cwd
    for _ in range(10):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError(
            f"No se encontró carpeta 'src/' desde {_cwd}. "
            f"Verifica que el notebook está dentro de AU_UJI/"
        )

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: ANÁLISIS DE NULOS — MÉTRICAS GLOBALES
# ============================================================================
# Calcula: total celdas nulas, % global, clasificación de columnas
# por severidad (sin nulos, bajo, medio, alto, muy alto).
# ============================================================================

print('=' * 60)
print('ANÁLISIS DE NULOS')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

n_filas = len(df)
n_cols = len(df.columns)

# --- Calcular nulos por columna ---
nulos_col = df.isnull().sum()
pct_nulos_col = (nulos_col / n_filas * 100)
total_celdas = n_filas * n_cols
total_nulos = nulos_col.sum()
pct_nulos_global = (total_nulos / total_celdas * 100)

# --- Lista de columnas con nulos (se usa en celdas posteriores) ---
cols_con_nulos = pct_nulos_col[pct_nulos_col > 0].index.tolist()

# --- Clasificar columnas por severidad ---
cols_sin_nulos = (pct_nulos_col == 0).sum()
cols_bajo = ((pct_nulos_col > 0) & (pct_nulos_col < 5)).sum()
cols_medio = ((pct_nulos_col >= 5) & (pct_nulos_col < 20)).sum()
cols_alto = ((pct_nulos_col >= 20) & (pct_nulos_col < 50)).sum()
cols_criticos = (pct_nulos_col >= 50).sum()

print(f'📊 Total celdas: {total_celdas:,}')
print(f'❓ Celdas nulas: {total_nulos:,} ({pct_nulos_global:.2f}%)')
print(f'✅ Columnas sin nulos: {cols_sin_nulos}')
print(f'⚠️ Columnas con nulos: {len(cols_con_nulos)}')

ANÁLISIS DE NULOS
📊 Total celdas: 4,054,016
❓ Celdas nulas: 294,706 (7.27%)
✅ Columnas sin nulos: 23
⚠️ Columnas con nulos: 14


In [3]:
# ============================================================================
# CELDA 3: GRÁFICOS — CLASIFICACIÓN Y BARRAS
# ============================================================================
# Gráfico 1: Donut con clasificación de columnas por severidad.
# Gráfico 2: Barras horizontales con % de nulos por columna.
# ============================================================================

print('=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

# --- Gráfico 1: Donut clasificación ---
clasif_labels = ['Sin nulos', 'Bajo (<5%)', 'Medio (5-20%)', 'Alto (20-50%)', 'Muy alto (>50%)']
clasif_valores = [cols_sin_nulos, cols_bajo, cols_medio, cols_alto, cols_criticos]
colores_clasif = ['#38a169', '#48bb78', '#ecc94b', '#ed8936', '#e53e3e']

# Filtrar categorías vacías
datos_donut = [(l, v, c) for l, v, c in zip(clasif_labels, clasif_valores, colores_clasif) if v > 0]

fig_donut = go.Figure(go.Pie(
    labels=[d[0] for d in datos_donut],
    values=[d[1] for d in datos_donut],
    hole=0.5,
    marker_colors=[d[2] for d in datos_donut],
    textinfo='label+value',
    textposition='outside'
))
fig_donut.update_layout(
    title='📊 Clasificación de Columnas por Nulos<br>'
              '<span style="font-size:11px; color:#718096;">'
              'Agrupadas por % de valores faltantes</span>',
    height=400,
    margin=dict(t=60, b=40, l=40, r=40),
    showlegend=False
)
fig_donut.write_html(RUTA_GRAFICOS / 'm03_donut_clasificacion.html', include_plotlyjs='cdn')
print('✅ Gráfico clasificación')

# --- Gráfico 2: Barras nulos por columna ---
nulos_ordenados = pct_nulos_col[pct_nulos_col > 0].sort_values(ascending=True)
colores_barras = [
    '#e53e3e' if v > 50 else '#ed8936' if v > 20 else '#ecc94b' if v > 5 else '#38a169'
    for v in nulos_ordenados.values
]

fig_barras = go.Figure(go.Bar(
    x=nulos_ordenados.values,
    y=nulos_ordenados.index,
    orientation='h',
    marker_color=colores_barras,
    text=[f'{v:.1f}%' for v in nulos_ordenados.values],
    textposition='outside'
))
fig_barras.update_layout(
    title='📊 % Nulos por Columna<br>'
          '<span style="font-size:11px; color:#718096;">'
          '🟢 <5% | 🟡 5-20% | 🟠 20-50% | 🔴 >50%</span>',
    xaxis_title='% Nulos',
    height=max(350, len(nulos_ordenados) * 25),
    margin=dict(t=80, b=50, l=180, r=80)
)
fig_barras.write_html(RUTA_GRAFICOS / 'm03_barras_nulos.html', include_plotlyjs='cdn')
print('✅ Gráfico barras nulos')

GENERANDO GRÁFICOS
✅ Gráfico clasificación
✅ Gráfico barras nulos


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS — PATRÓN Y CORRELACIÓN DE NULOS
# ============================================================================
# Gráfico 3: Heatmap binario del patrón de nulidad.
#   - Columnas ordenadas por % de nulos (mayor a menor)
#   - Colores: gris claro = presente, azul oscuro = nulo
#   - Muestra 200 filas aleatorias
# Gráfico 4: Correlación entre columnas nulas.
#   - Paleta coolwarm suave, centrada en 0
#   - Solo muestra valores con |corr| >= 0.3
#   - Detecta qué columnas son nulas simultáneamente
# ============================================================================

if len(cols_con_nulos) > 0:
    # --- Ordenar columnas por % de nulos (mayor a menor) ---
    cols_ordenadas = pct_nulos_col[cols_con_nulos].sort_values(ascending=False).index.tolist()

    # --- Gráfico 3: Patrón de nulos (muestra de filas) ---
    muestra = df[cols_ordenadas].isnull().astype(int).sample(min(200, n_filas), random_state=42)

    fig_patron = go.Figure(go.Heatmap(
        z=muestra.values,
        x=muestra.columns,
        y=list(range(len(muestra))),
        colorscale=[
            [0, '#E5E5E5'],   # Presente: gris claro
            [1, '#1565C0']    # Nulo: azul oscuro
        ],
        showscale=False,
        hovertemplate='Columna: %{x}<br>Fila: %{y}<br>Nulo: %{z}<extra></extra>'
    ))
    fig_patron.update_layout(
        title='🔍 Patrón de Nulos (200 filas, 1=nulo, 0=presente)<br>'
              '<span style="font-size:11px; color:#718096;">'
              'Columnas ordenadas por % nulos (mayor→menor) | '
              '⬜ Presente | 🟦 Nulo</span>',
        height=450,
        margin=dict(t=90, b=100, l=40, r=40),
        xaxis_tickangle=45,
        yaxis=dict(showticklabels=False),
        plot_bgcolor='white'
    )
    fig_patron.write_html(RUTA_GRAFICOS / 'm03_patron_nulos.html', include_plotlyjs='cdn')
    print('✅ Gráfico patrón nulos (colores mejorados, columnas ordenadas)')

    # --- Gráfico 4: Correlación de nulos ---
    if len(cols_con_nulos) >= 2:
        matriz_bool = df[cols_ordenadas].isnull()
        corr_nulos = matriz_bool.corr()

        # Texto: solo mostrar valores con |corr| >= 0.3 (el resto vacío)
        texto_corr = []
        for fila in corr_nulos.values:
            fila_texto = []
            for val in fila:
                if abs(val) >= 0.3:
                    fila_texto.append(f'{val:.1f}')
                else:
                    fila_texto.append('')
            texto_corr.append(fila_texto)

        fig_corr = go.Figure(go.Heatmap(
            z=corr_nulos.values,
            x=corr_nulos.columns,
            y=corr_nulos.index,
            colorscale=[
                [0, '#4393C3'],    # Azul suave (inversa)
                [0.25, '#92C5DE'],
                [0.5, '#F7F7F7'],  # Blanco (sin correlación)
                [0.75, '#F4A582'],
                [1, '#D6604D']     # Rojo suave (simultáneos)
            ],
            zmid=0,
            text=texto_corr,
            texttemplate='%{text}',
            textfont={'size': 10},
            hovertemplate='%{x} vs %{y}<br>Corr: %{z:.2f}<extra></extra>'
        ))
        fig_corr.update_layout(
            title='🔗 Correlación de Nulos entre Columnas<br>'
                  '<span style="font-size:11px; color:#718096;">'
                  '🔴 Se ausentan juntos | 🔵 Inversos | Solo muestra |corr| ≥ 0.3</span>',
            height=500,
            margin=dict(t=90, b=120, l=170, r=40),
            xaxis_tickangle=45,
            plot_bgcolor='white'
        )
        fig_corr.write_html(RUTA_GRAFICOS / 'm03_correlacion_nulos.html', include_plotlyjs='cdn')
        print('✅ Gráfico correlación nulos (paleta suave, solo |corr| >= 0.3)')
else:
    print('✅ Sin columnas con nulos — no se generan gráficos de patrón/correlación')

✅ Gráfico patrón nulos (colores mejorados, columnas ordenadas)
✅ Gráfico correlación nulos (paleta suave, solo |corr| >= 0.3)


In [5]:
# ============================================================================
# CELDA 5: TABLA DETALLADA DE COLUMNAS CON NULOS
# ============================================================================
# Genera tabla con: columna, tipo, nulos absolutos, %, clasificación.
# Ordenada de mayor a menor porcentaje de nulos.
# ============================================================================

print('\n📋 Preparando tabla detallada...')

cols_nulos_detalle = []
for col in cols_con_nulos:
    n_nulos = nulos_col[col]
    pct = pct_nulos_col[col]
    dtype = str(df[col].dtype)

    # Clasificar severidad
    if pct >= 50:
        clasif = '🔴 Muy alto (>50%)'
    elif pct >= 20:
        clasif = '🟠 Alto (20-50%)'
    elif pct >= 5:
        clasif = '🟡 Medio (5-20%)'
    else:
        clasif = '🟢 Bajo (<5%)'

    cols_nulos_detalle.append({
        'Columna': col,
        'Tipo': dtype,
        'Nulos': n_nulos,
        '%': pct,
        'Clasificación': clasif
    })

df_detalle = pd.DataFrame(cols_nulos_detalle).sort_values('%', ascending=False)
print(f'✅ Tabla preparada: {len(df_detalle)} columnas con nulos')


📋 Preparando tabla detallada...
✅ Tabla preparada: 14 columnas con nulos


In [6]:
# ============================================================================
# CELDA 6: GENERAR HTML — PÁGINA M03 NULOS
# ============================================================================
# Ensambla la página con:
# - KPIs: % nulos global, cols sin/con nulos, celdas nulas
# - Fila 1: Donut clasificación + Barras %
# - Fila 2: Patrón nulos + Correlación nulos
# - Tabla detallada
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Navegación dinámica ---
nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa="fase2", modulo_activo="m03"
)

# --- KPIs ---
KPIS = [
    {'valor': f'{pct_nulos_global:.1f}%', 'titulo': '% Nulos Global'},
    {'valor': str(cols_sin_nulos), 'titulo': 'Cols Sin Nulos'},
    {'valor': str(len(cols_con_nulos)), 'titulo': 'Cols Con Nulos'},
    {'valor': formato_numero_es(total_nulos), 'titulo': 'Celdas Nulas'},
]
kpis_html = generar_kpis_html(KPIS)

# --- Fila 1: Donut + Barras ---
seccion_fila1 = generar_seccion_html(
    titulo="Clasificación y Distribución",
    contenido='''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m03_donut_clasificacion.html" width="100%" height="420" frameborder="0"></iframe>
            <iframe src="graficos/m03_barras_nulos.html" width="100%" height="420" frameborder="0"></iframe>
        </div>
    ''',
    icono="📊"
)

# --- Fila 2: Patrón + Correlación ---
seccion_fila2 = generar_seccion_html(
    titulo="Patrón y Correlación",
    contenido='''
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m03_patron_nulos.html" width="100%" height="470" frameborder="0"></iframe>
            <iframe src="graficos/m03_correlacion_nulos.html" width="100%" height="470" frameborder="0"></iframe>
        </div>
    ''',
    icono="🔍"
)

# --- Tabla detallada ---
tabla_html = df_detalle.to_html(classes='tabla-estadisticas', index=False, float_format=lambda x: f'{x:.2f}')
seccion_tabla = generar_seccion_html(
    titulo="Detalle por Columna",
    contenido=f'<div style="overflow-x:auto; max-height:400px; overflow-y:auto;">{tabla_html}</div>',
    icono="📋"
)

# --- Ensamblar y guardar ---
contenido_html = kpis_html + seccion_fila1 + seccion_fila2 + seccion_tabla

html_completo = render_base_html(
    titulo="M03: Nulos",
    icono="❓",
    subtitulo="Fase 2: EDA Datos Originales | TFM Abandono Universitario",
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f2_m03_nulos.ipynb',
    notebook_carpeta='fase2_eda'
)

ruta_html = RUTA_FASE2 / "m03_nulos.html"
guardar_html(html_completo, ruta_html)
print(f"\n✅ HTML: {ruta_html}")

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m03_nulos.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m03_nulos.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN DE EJECUCIÓN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M03 COMPLETADO')
print('=' * 60)
print('📊 Fila 1: Donut clasificación + Barras %')
print('📊 Fila 2: Patrón nulos + Correlación nulos')
print('📋 Tabla: Detalle columnas con nulos')
print('📌 Siguiente: f2_m04_univariante_num.ipynb')


✅ F2-M03 COMPLETADO
📊 Fila 1: Donut clasificación + Barras %
📊 Fila 2: Patrón nulos + Correlación nulos
📋 Tabla: Detalle columnas con nulos
📌 Siguiente: f2_m04_univariante_num.ipynb
